# Northpeak Supply: predicting late invoice payments (starter)

Before you start, run `01_setup_and_generate_data.sql` in a SQL worksheet. It creates `NORTHPEAK_ML_WH`, the `NORTHPEAK_ML.CREDIT` schema and the `CUSTOMERS`, `INVOICES` and `PAYMENTS` tables.

Create this notebook on `NORTHPEAK_ML.CREDIT` with the query warehouse `NORTHPEAK_ML_WH`, and add the packages `snowflake-ml-python`, `xgboost` and `scikit-learn` (they are also listed in `environment.yml`).

An invoice is **late** when `PAYMENTS.DAYS_LATE > 0`.

Run the setup cell first, then one cell per task.

In [ ]:
# Setup (given): grab the notebook's session and point it at the lab's objects.
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark import Window

session = get_active_session()
session.use_warehouse("NORTHPEAK_ML_WH")
session.use_database("NORTHPEAK_ML")
session.use_schema("CREDIT")

for table in ["CUSTOMERS", "INVOICES", "PAYMENTS"]:
    print(f"{table}: {session.table(table).count()} rows")

## Task 1: Explore the data and find what drives late payment

Look at the late rate across the columns you have: customer segment, payment terms, autopay, early-payment discount, channel, invoice size, month, customer tenure.

Hint: Snowpark DataFrames (Section 7), or plain SQL in a worksheet or with `session.sql(...)`.

In [ ]:
# TODO: Compute the overall late rate.
# TODO: Compare the late rate across the columns that might drive it.
# TODO: Note which drivers move the late rate the most.

## Task 2: Find the column that gives the answer away, and keep it out

Some columns are only known after the invoice is paid, or late. A model trained on them looks brilliant in testing and is useless on the day an invoice is issued.

Hint: for every column, ask "would I know this on the invoice date?"

In [ ]:
# TODO: Find the column whose value depends on what happened after the invoice was issued.
# TODO: Show with numbers why it leaks the answer.
# TODO: Write down which columns you will keep out of the features, and why.

## Task 3: Build features using only what was known on the invoice date

Build one row per invoice with your features, the label `PAID_LATE` (1 or 0) and `INVOICE_DATE`, and save it as a table named `INVOICE_FEATURES`.

Features from a customer's history may only use invoices that came before this one, and a payment track record may only count invoices that were already paid by this invoice's date.

Hint: Snowpark DataFrame API, window functions and feature engineering (Section 7).

In [ ]:
# TODO: Join INVOICES, CUSTOMERS and PAYMENTS, and build the PAID_LATE label.
# TODO: Add invoice-level and customer-level features, and encode the categories.
# TODO: Add history features over each customer's EARLIER invoices only.
# TODO: Add a point-in-time payment track record, including a PRIOR_LATE_RATE
#       column (the share of the customer's already-paid earlier invoices that
#       were late). Task 5 needs it.
# TODO: Save the result as INVOICE_FEATURES.

## Task 4: Train a model and test it on invoices it has never seen

The test set should look like the future the model will face in production.

Hint: train, score and register a model (Section 7). Think about whether a random split is honest for data that has a date on every row.

In [ ]:
# TODO: Split INVOICE_FEATURES into a training set and a test set.
# TODO: Train an XGBoost classifier on the training set.
# TODO: Keep the model, the test set, the feature list and the label name in
#       variables, because the next two cells use them.

## Task 5: Prove it beats the rule collections uses today

Today the collections team flags an invoice when the customer paid at least half of their earlier invoices late. Compare your model against that rule, and against simply predicting "on time" for every invoice, on the same test set.

Hint: ROC AUC, precision, recall and F1 from scikit-learn. Also think like the business: if the team can only call the riskiest 20% of invoices, how many late ones does each approach catch?

In [ ]:
# TODO: Score the test set with your model.
# TODO: Build the two baselines: always on time, and the late-before rule.
# TODO: Compare all three with the same metrics, in one table.
# TODO: Compare how many late invoices each approach catches in its top 20%.

## Task 6: Register the model so it's ready to score new invoices

Hint: the Snowflake Model Registry (Section 7). Name it `LATE_PAYMENT_MODEL`, version `V1`.

In [ ]:
# TODO: Log the model with its test metrics, a comment and a sample of the input.
# TODO: Load the registered version back and check it gives the same predictions
#       as the model in memory.
# TODO: Confirm the model and its version from SQL with SHOW MODELS.

## Cleanup (optional)

When you are done, end the notebook session, then run this in a SQL worksheet:

```sql
DROP DATABASE IF EXISTS NORTHPEAK_ML;
DROP WAREHOUSE IF EXISTS NORTHPEAK_ML_WH;
```

Dropping the database also removes the notebook, the feature table and the registered model.